# Component 2 — Treatment-Aware Counterfactual Fairness (Synthetic Example)

This notebook:
1) Generates synthetic data with a treatment variable influenced by covariates and group membership.
2) Fits outcome and treatment models (LightGBM).
3) Runs Component 2 fairness estimation.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from lightgbm import LGBMClassifier

!pip install --force-reinstall --no-deps --no-cache-dir \
    git+https://github.com/vsubbian/FairLogue.git

import FairLogue

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def make_synth_component2_standard(n=12000, seed=21):
    rng = np.random.default_rng(seed)

    # Protected attributes (binary for simplicity)
    # A1 ~ race-like axis, A2 ~ sex-like axis
    A1 = rng.integers(0, 2, size=n)  # 0/1
    A2 = rng.integers(0, 2, size=n)  # 0/1

    # Covariates used in propensity model (pi_xvars)
    X_1 = rng.normal(0, 1, size=n)
    X_2 = rng.normal(0, 1, size=n)
    X_3 = rng.normal(0, 1, size=n)
    X_4 = rng.normal(0, 1, size=n)

    # Risk score model output (S_prob): predicted probability from some classifier.
    # In real use, S_prob is your model's predicted probability; here we simulate it.
    s_logit = (
        -0.3
        + 0.6 * X_1
        - 0.4 * X_2
        + 0.25 * X_3
        + 0.15 * X_4
        + 0.20 * A1
        - 0.10 * A2
        + 0.15 * (A1 * A2)
    )
    S_prob = sigmoid(s_logit)

    # Treatment assignment D depends on A1, A2, covariates, and S_prob
    # This creates a "treatment disparity" mechanism that Component 2 is designed to handle.
    d_logit = (
        -0.8
        + 0.5 * A1
        - 0.3 * A2
        + 0.35 * (A1 * A2)
        + 0.7 * S_prob
        + 0.4 * X_1
        + 0.2 * X_2
        - 0.1 * X_3
        + 0.1 * X_4
    )
    pi_true = sigmoid(d_logit)
    D = rng.binomial(1, pi_true, size=n)

    # Potential outcome under no treatment Y0 (binary)
    y0_logit = (
        -1.2
        + 0.9 * S_prob
        + 0.25 * X_1
        - 0.15 * X_2
        + 0.15 * X_3
        + 0.10 * X_4
        + 0.10 * A1
        + 0.05 * A2
    )
    p_y0 = sigmoid(y0_logit)
    Y0 = rng.binomial(1, p_y0, size=n)

    # Treatment effect: treatment reduces the probability of Y=1
    # Create Y1 by shifting the logit downward.
    treat_effect = 0.8
    y1_logit = y0_logit - treat_effect
    p_y1 = sigmoid(y1_logit)
    Y1 = rng.binomial(1, p_y1, size=n)

    # Observed outcome
    Y = np.where(D == 1, Y1, Y0).astype(int)

    df = pd.DataFrame({
        "A1": A1.astype(int),
        "A2": A2.astype(int),
        "X_1": X_1,
        "X_2": X_2,
        "X_3": X_3,
        "X_4": X_4,
        "S_prob": S_prob,
        "D": D.astype(int),
        "Y": Y.astype(int),
    })
    return df

df = make_synth_component2_standard(n=12000, seed=21)
df.head()

In [ ]:
pi_xvars = ["X_1", "X_2", "X_3", "X_4"]

pi_model = smf.glm(
    formula="D ~ A1*A2 + A1 + A2 + S_prob + X_1 + X_2 + X_3 + X_4",
    data=df,
    family=sm.families.Binomial()
).fit()

pi_model.summary().tables[0]

In [ ]:
results = Fairlogue.Component2.analysis_estimation(
    data=df,
    cutoff=0.5,
    estimator_type="standard",
    pi_model=pi_model,
    pi_model_type="glm",
    pi_xvars=pi_xvars,
    gen_null=True,
    bootstrap="rescaled",
    R_null=200,   # reduce for speed in a demo
    B=200,        # reduce for speed in a demo
)

# results is a dict; keys include defs, est_choice, and optionally table_null / boot_out
results.keys()

In [ ]:
sampsize = len(df)

est_summaries, table_null_delta, table_uval = FairLogue.Component1.get_plots(
    results,
    sampsize=sampsize,
    alpha=0.05,
    m_factor=0.75,
    delta_uval=0.10,
)

est_summaries.head(20)

In [ ]:
table_uval